In [ ]:
import os
import shutil
import subprocess
import sys

env_path = "/tmp/repos-env"
python_bin = os.path.join(env_path, "bin", "python")
conda_bin = shutil.which("mamba") or shutil.which("conda")

# Required libraries and C++ dependencies
conda_packages = [
    "python-gmsh",
    "meshio",
    "nibabel",
    "pynrrd",
    "scikit-image",
    "scikit-learn",
    "scipy",
    "numpy",
    "vtk",
    "trimesh",
    "rtree",      # spatial index required by trimesh ray queries
    "ipykernel",
]

# conda package name -> import name (only where they differ)
import_names = {
    "python-gmsh": "gmsh",
    "pynrrd": "nrrd",
    "scikit-image": "skimage",
    "scikit-learn": "sklearn",
}


def missing_packages():
    missing = []
    for pkg in conda_packages:
        mod = import_names.get(pkg, pkg)
        result = subprocess.run(
            [python_bin, "-c", f"import {mod}"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        if result.returncode != 0:
            missing.append(pkg)
    return missing


if not os.path.exists(env_path):
    print("Creating environment in /tmp/repos-env...")
    cmd = [conda_bin, "create", "-p", env_path, "-c", "conda-forge", "-y"] + conda_packages
    subprocess.check_call(cmd)

    # Register kernel with Jupyter
    kernel_cmd = [
        python_bin, "-m", "ipykernel", "install", "--user",
        "--name", "repos-env",
        "--display-name", "Python (repos-env)",
    ]
    subprocess.check_call(kernel_cmd)
    print("\nEnvironment build complete!")
else:
    print("Environment already exists in /tmp/repos-env.")
    missing = missing_packages()
    if missing:
        print(f"Installing missing packages: {', '.join(missing)}")
        cmd = [conda_bin, "install", "-p", env_path, "-c", "conda-forge", "-y"] + missing
        subprocess.check_call(cmd)
        print("Done. Restart the kernel to pick up the new packages.")
    else:
        print("All packages present.")